In [ ]:
import requests, time, re, pickle, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')
print("✅ TensorFlow:", tf.__version__)
print("✅ All imports done!")


In [ ]:
API_KEY  = "310dfba3252d94966cb65a063f8174058108"
BASE_URL = "https://api.fda.gov/drug/label.json"

drug_terms = [
    "ibuprofen", "aspirin", "paracetamol",
    "metformin", "amoxicillin", "insulin",
    "omeprazole", "atorvastatin", "azithromycin",
    "cetirizine", "diclofenac", "prednisone", "salbutamol"
]
print(f"📋 {len(drug_terms)} drugs:", drug_terms)


In [ ]:
def fetch_medical_data(terms, target_size=2000):
    all_data = []
    print("🚀 Collecting data...")
    while len(all_data) < target_size:
        for term in tqdm(terms, desc="Fetching"):
            url = f"{BASE_URL}?search=openfda.generic_name:{term}&limit=100"
            r   = requests.get(url)
            if r.status_code != 200:
                continue
            for item in r.json().get("results", []):
                all_data.append({
                    "drug"    : term,
                    "text"    : " ".join(item.get("description", [""])),
                    "purpose" : " ".join(item.get("purpose",     [""])),
                    "warnings": " ".join(item.get("warnings",    [""]))
                })
                if len(all_data) >= target_size:
                    break
            time.sleep(0.2)
            if len(all_data) >= target_size:
                break
    df = pd.DataFrame(all_data)
    print(f"\n✅ Collected: {len(df)} records")
    print(df["drug"].value_counts())
    return df

df = fetch_medical_data(drug_terms, target_size=2000)
df.head(3)


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+",         " ", text)
    return text.strip()

df["clean_text"] = (
    df["text"].fillna("") + " " +
    df["purpose"].fillna("") + " " +
    df["warnings"].fillna("")
)
df["clean_text"] = df["clean_text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 10].reset_index(drop=True)

print(f"✅ After cleaning: {len(df)} records")
print("\nSample:")
print(df["clean_text"].iloc[0][:300])


In [ ]:
VOCAB_SIZE    = 5000
MAX_LEN       = 120
EMBEDDING_DIM = 64

# Label encoding
le = LabelEncoder()
df["label"] = le.fit_transform(df["drug"])
NUM_CLASSES  = len(le.classes_)
print(f"✅ {NUM_CLASSES} classes:", list(le.classes_))

# Tokenize
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"])
sequences = tokenizer.texts_to_sequences(df["clean_text"])

X = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")
y = to_categorical(df["label"], num_classes=NUM_CLASSES)

print(f"\n✅ X shape: {X.shape}")
print(f"✅ y shape: {y.shape}")


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train : {X_train.shape[0]}")
print(f"Val   : {X_val.shape[0]}")
print(f"Test  : {X_test.shape[0]}")


In [ ]:
def build_lstm(vocab_size, embedding_dim, max_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_len, name="embedding"),
        Bidirectional(LSTM(128, return_sequences=True), name="biLSTM_1"),
        Dropout(0.3),
        LSTM(64, name="LSTM_2"),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(num_classes, activation="softmax", name="output")
    ], name="LSTM_Drug_Classifier")

    model.compile(optimizer="adam",
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

lstm_model = build_lstm(VOCAB_SIZE, EMBEDDING_DIM, MAX_LEN, NUM_CLASSES)
lstm_model.summary()


In [ ]:
lstm_callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint("best_lstm.h5", monitor="val_accuracy",
                    save_best_only=True, verbose=1)
]

lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=lstm_callbacks
)
print("\n✅ LSTM Training complete!")


In [ ]:
def plot_history(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history.history["accuracy"],     label="Train", color="royalblue")
    axes[0].plot(history.history["val_accuracy"], label="Val",   color="tomato")
    axes[0].set_title(f"Accuracy — {title}"); axes[0].legend(); axes[0].grid(alpha=.3)

    axes[1].plot(history.history["loss"],     label="Train", color="royalblue")
    axes[1].plot(history.history["val_loss"], label="Val",   color="tomato")
    axes[1].set_title(f"Loss — {title}"); axes[1].legend(); axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

plot_history(lstm_history, "LSTM Baseline")


In [ ]:
def evaluate_model(model, X_test, y_test, label_encoder, model_name="Model"):
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    y_pred     = model.predict(X_test, verbose=0)
    y_pred_cls = np.argmax(y_pred, axis=1)
    y_true_cls = np.argmax(y_test, axis=1)

    print(f"\n{'='*50}")
    print(f"📊 {model_name} Results")
    print(f"{'='*50}")
    print(f"🎯 Test Accuracy : {acc*100:.2f}%")
    print(f"📉 Test Loss     : {loss:.4f}")
    print(f"\n{classification_report(y_true_cls, y_pred_cls, target_names=label_encoder.classes_)}")

    # Confusion matrix
    cm = confusion_matrix(y_true_cls, y_pred_cls)
    plt.figure(figsize=(11, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title(f"Confusion Matrix — {model_name}", fontsize=13)
    plt.ylabel("True"); plt.xlabel("Predicted")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()
    return acc

lstm_acc = evaluate_model(lstm_model, X_test, y_test, le, "LSTM Baseline")


In [ ]:
from tensorflow.keras.layers import GRU

def build_gru(vocab_size, embedding_dim, max_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_len),
        Bidirectional(GRU(128, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(GRU(64)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ], name='GRU_Drug_Classifier')

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

gru_model = build_gru(VOCAB_SIZE, EMBEDDING_DIM, MAX_LEN, NUM_CLASSES)
gru_model.summary()
